# Лабораторная работа № 1. Массивы NumPy и подготовка данных

**Кейс:** аудит данных интернет-магазина.
**Ограничения:** в обязательных заданиях нет циклов `for`/`while`, нет Pandas, нет ручного перебора. Используются только векторизованные операции NumPy и булевы маски.

In [1]:
import numpy as np

order_id = np.array([1001, 1002, 1003, 1004, 1005, 1006])
quantity = np.array([2, 3, 0, 5, -1, 4])
unit_price = np.array([1500, 2000, 1200, 3000, 2500, np.nan])
discount = np.array([0.10, 0.05, 0.00, 0.15, 0.10, 0.20])

## Задание 1. Проверка структуры

In [2]:
def describe(name, a):
    print(f"{name:11s} ndim={a.ndim} shape={a.shape} size={a.size} dtype={a.dtype}")

describe("order_id", order_id)
describe("quantity", quantity)
describe("unit_price", unit_price)
describe("discount", discount)

print("Все формы одинаковы:", order_id.shape == quantity.shape == unit_price.shape == discount.shape)

order_id    ndim=1 shape=(6,) size=6 dtype=int64
quantity    ndim=1 shape=(6,) size=6 dtype=int64
unit_price  ndim=1 shape=(6,) size=6 dtype=float64
discount    ndim=1 shape=(6,) size=6 dtype=float64
Все формы одинаковы: True


**Вывод.** Все четыре массива одномерные, форма `(6,)`, по 6 элементов. Значит, по индексам их сопоставлять технически можно: элемент с индексом `i` в каждом массиве описывает один и тот же заказ (при условии, что данные записывались синхронно, что подтверждается только общим ключом `order_id`).
`unit_price` имеет тип `float64`, хотя цены целые: причина в `np.nan`, который существует только для float. Остальные массивы пропусков не содержат.

## Задание 2. Контроль качества

In [3]:
bad_qty = quantity <= 0
bad_price = np.isnan(unit_price)
bad_discount = (discount < 0) | (discount > 1)
failed = bad_qty | bad_price | bad_discount

print("bad_qty      :", bad_qty, "-> заказы", order_id[bad_qty])
print("bad_price    :", bad_price, "-> заказы", order_id[bad_price])
print("bad_discount :", bad_discount, "-> заказы", order_id[bad_discount])
print("failed (any) :", failed, "-> заказы", order_id[failed])
print(f"Не прошли проверку: {failed.sum()} из {failed.size} ({failed.mean():.0%})")

bad_qty      : [False False  True False  True False] -> заказы [1003 1005]
bad_price    : [False False False False False  True] -> заказы [1006]
bad_discount : [False False False False False False] -> заказы []
failed (any) : [False False  True False  True  True] -> заказы [1003 1005 1006]
Не прошли проверку: 3 из 6 (50%)


**Вывод.**
- Количество `<= 0`: заказы **1003** (ноль) и **1005** (`-1`).
- Пропущенная цена: заказ **1006**.
- Скидок вне диапазона `[0, 1]` нет.
- Первичную проверку не проходят **3 из 6 заказов (50 %)**. Для финансовых расчётов исходные данные использовать нельзя без очистки.

## Задание 3. Формирование рабочего набора

In [4]:
valid = (
    (quantity > 0)
    & np.isfinite(unit_price) & (unit_price >= 0)
    & (discount >= 0) & (discount <= 1)
)
print("valid:", valid)

order_id_c = order_id[valid]
quantity_c = quantity[valid]
price_c = unit_price[valid]
discount_c = discount[valid]

print("order_id:", order_id_c)
print("quantity:", quantity_c)
print("price   :", price_c)
print("discount:", discount_c)

valid: [ True  True False  True False False]
order_id: [1001 1002 1004]
quantity: [2 3 5]
price   : [1500. 2000. 3000.]
discount: [0.1  0.05 0.15]


**Почему нельзя фильтровать каждый массив независимо.** Покажем, что получится, если для каждого массива применить только «свою» проверку:

In [5]:
qty_only = quantity[quantity > 0]
price_only = unit_price[np.isfinite(unit_price)]
print("quantity (qty > 0)  :", qty_only, "длина", qty_only.size)
print("price (finite)      :", price_only, "длина", price_only.size)

quantity (qty > 0)  : [2 3 5 4] длина 4
price (finite)      : [1500. 2000. 1200. 3000. 2500.] длина 5


**Вывод.** Длины стали разными (4 и 5), а даже при случайно одинаковой длине порядок элементов уже не соответствует друг другу: цена заказа 1003 (`1200`) осталась бы в массиве цен, а его количество (`0`) удалено. Все последующие пары «количество - цена» сдвинулись бы, и выручка считалась бы для несуществующих комбинаций. Поэтому фильтровать нужно **одной общей маской**, применяемой ко всем массивам: так строка `i` по-прежнему описывает один и тот же заказ.

## Задание 4. Подготовка матрицы

In [6]:
X = np.column_stack((quantity_c, price_c, discount_c))
print(X)
print("shape:", X.shape)

[[2.0e+00 1.5e+03 1.0e-01]
 [3.0e+00 2.0e+03 5.0e-02]
 [5.0e+00 3.0e+03 1.5e-01]]
shape: (3, 3)


**Вывод.** Форма `(3, 3)`: **строка = заказ, столбец = признак** (количество, цена, скидка). Выбрана классическая организация «наблюдения × признаки»:
- каждый столбец - однородная величина, его легко взять срезом (`X[:, 1]` - все цены);
- агрегаты по признакам считаются через `axis=0`, по заказам через `axis=1`;
- векторные формулы записываются без циклов, например выручка `X[:, 0] * X[:, 1] * (1 - X[:, 2])`.

Замечание: NumPy привёл все столбцы к общему типу `float64`, поэтому количество стало `2.0`, `3.0`, `5.0`.

## Задание 5. Преобразование данных

In [7]:
sales = np.arange(1, 25)          # 24 значения: продажи за 4 недели x 6 категорий
m_4x6 = sales.reshape(4, 6)
m_6x4 = sales.reshape(6, 4)

print("4 x 6 (недели x категории):")
print(m_4x6)
print("\n6 x 4 через reshape:")
print(m_6x4)
print("\nтранспонирование .T:")
print(m_4x6.T)
print("\nreshape(6, 4) совпадает с .T ?", np.array_equal(m_6x4, m_4x6.T))

4 x 6 (недели x категории):
[[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]
 [13 14 15 16 17 18]
 [19 20 21 22 23 24]]

6 x 4 через reshape:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]
 [17 18 19 20]
 [21 22 23 24]]

транспонирование .T:
[[ 1  7 13 19]
 [ 2  8 14 20]
 [ 3  9 15 21]
 [ 4 10 16 22]
 [ 5 11 17 23]
 [ 6 12 18 24]]

reshape(6, 4) совпадает с .T ? False


**Вывод.** `reshape` не «поворачивает» таблицу, а лишь заново нарезает ту же линейную последовательность `1..24` на строки новой длины.
- В матрице `4 × 6` строка - неделя, столбец - категория: первая строка `[1..6]` - все категории первой недели.
- В `6 × 4` через `reshape` строка `[1, 2, 3, 4]` уже захватывает четыре категории одной недели, а следующая строка начинается с остатка: значения разных недель и категорий перемешаны. Если считать, что строки - категории, а столбцы - недели, то смысл данных **потерян**.
- Чтобы поменять роли осей, не искажая данные, нужно **транспонирование** `.T` (результат отличается от `reshape(6, 4)`, что подтверждает проверка выше).

Значит, форма массива - это соглашение о смысле осей; его надо зафиксировать до расчётов.

## Задание 6. Проверка источников
Моделируем загрузку из разных файлов: заказы и цены. Рассмотрим три ситуации.

In [8]:
ids_orders = np.array([1001, 1002, 1003, 1004, 1005, 1006])

# A: в файле цен потеряна запись 1003
ids_prices_a = np.array([1001, 1002, 1004, 1005, 1006])
prices_a = np.array([1500, 2000, 3000, 2500, 3100])

print("A. длины равны?", ids_orders.size == ids_prices_a.size)
print("   нет в ценах   :", np.setdiff1d(ids_orders, ids_prices_a))
print("   нет в заказах :", np.setdiff1d(ids_prices_a, ids_orders))

A. длины равны? False
   нет в ценах   : [1003]
   нет в заказах : []


In [9]:
# B: длины равны (6 и 6), но потеряна 1003 и появилась лишняя 1007
ids_prices_b = np.array([1001, 1002, 1004, 1005, 1006, 1007])

print("B. длины равны?          ", ids_orders.size == ids_prices_b.size)
print("   поэлементное сравнение:", ids_orders == ids_prices_b)
print("   array_equal           :", np.array_equal(ids_orders, ids_prices_b))
print("   нет в ценах           :", np.setdiff1d(ids_orders, ids_prices_b))
print("   лишние в ценах        :", np.setdiff1d(ids_prices_b, ids_orders))

B. длины равны?           True
   поэлементное сравнение: [ True  True False False False False]
   array_equal           : False
   нет в ценах           : [1003]
   лишние в ценах        : [1007]


In [10]:
# C: те же идентификаторы, но другой порядок
ids_shuffled = np.array([1002, 1001, 1003, 1004, 1005, 1006])

print("C. множества ключей совпадают:", np.array_equal(np.sort(ids_orders), np.sort(ids_shuffled)))
print("   порядок совпадает         :", np.array_equal(ids_orders, ids_shuffled))

C. множества ключей совпадают: True
   порядок совпадает         : False


**Какие проверки нужны до сопоставления:**
1. сравнить `shape` / `size` обоих массивов;
2. убедиться, что ключи уникальны (`np.unique(ids).size == ids.size`);
3. сравнить множества ключей в обе стороны (`np.setdiff1d`, `np.isin`);
4. проверить порядок (`np.array_equal`) либо сопоставлять по ключу, а не по позиции;
5. проверить `dtype` и наличие `NaN` / `inf` в значениях.

Корректное сопоставление по ключу для сценария A (там, где цены нет, ставим `NaN`):

In [11]:
pos = np.clip(np.searchsorted(ids_prices_a, ids_orders), 0, ids_prices_a.size - 1)
found = ids_prices_a[pos] == ids_orders
joined = np.where(found, prices_a[pos], np.nan)
print(joined)

[1500. 2000.   nan 3000. 2500. 3100.]


**Вывод.** Одинаковое количество элементов не гарантирует правильного сопоставления: в сценарии B длины равны, но одна запись потеряна, а другая лишняя, поэтому все цены после позиции 2 приписались бы не тем заказам. Сценарий C показывает ту же проблему из-за порядка. Ошибка опасна тем, что **не вызывает исключения** и даёт правдоподобные, но неверные числа. Сопоставлять нужно по ключу `order_id`.

## Задание 7. Подготовка к аналитике
Матрица продаж: 4 региона (строки) × 6 недель (столбцы).

In [12]:
n_regions, n_weeks = 4, 6
rng = np.random.default_rng(42)
region_sales = rng.integers(10_000, 50_000, size=(n_regions, n_weeks)).astype(float)
print(region_sales)

print("\nнет NaN        :", not np.isnan(region_sales).any())
print("нет inf        :", not np.isinf(region_sales).any())
print("все конечные   :", np.isfinite(region_sales).all())
print("нет отрицат.   :", not (region_sales < 0).any())
print("форма (4, 6)   :", region_sales.shape == (n_regions, n_weeks))

assert region_sales.shape == (n_regions, n_weeks)
assert np.isfinite(region_sales).all() and (region_sales >= 0).all()

[[13570. 40958. 36182. 27555. 27320. 44343.]
 [13437. 37894. 18058. 13767. 31059. 49024.]
 [39430. 40445. 38699. 41442. 30529. 15124.]
 [43589. 28015. 30014. 24831. 17301. 47070.]]

нет NaN        : True
нет inf        : True
все конечные   : True
нет отрицат.   : True
форма (4, 6)   : True


Проверим, что контроль действительно ловит ошибки, на испорченной копии:

In [13]:
broken = region_sales.copy()
broken[1, 2] = np.nan
broken[3, 4] = -500
print("NaN:", np.isnan(broken).sum(), "| отрицательных:", (broken < 0).sum(),
      "| все конечные:", np.isfinite(broken).all())

NaN: 1 | отрицательных: 1 | все конечные: False


**Вывод.** Исходная матрица `4 × 6` не содержит пропусков, бесконечностей и отрицательных продаж, размер совпадает с числом регионов и недель, поэтому она пригодна для агрегатов по регионам (`axis=1`) и неделям (`axis=0`). Испорченная копия показывает, что проверки срабатывают. Данные сгенерированы случайно (`seed=42`), реальных продаж они не отражают.

## Задание 8. Аналитическое заключение

**Критические проблемы**

| Проблема | Где | Последствие для анализа |
|---|---|---|
| Пропущенная цена (`NaN`) | заказ 1006 | выручка по заказу = `NaN`; `sum` / `mean` по столбцу тоже `NaN`, итоги ломаются |
| Отрицательное количество | заказ 1005 | неясно, возврат это или ошибка ввода; занижает выручку и искажает средние |
| Нет гарантии сопоставления источников | заказы vs цены | молчаливое присвоение цен чужим заказам, самая опасная ошибка |

**Некритические проблемы**

| Проблема | Где | Последствие |
|---|---|---|
| Нулевое количество | заказ 1003 | выручка 0; достаточно исключить или пометить, влияет на число заказов и среднее количество |
| Смешанные типы (`int` и `float`) | `unit_price`, матрица `X` | автоматическое приведение к `float64`, учитывать при чтении результатов |
| Высокая скидка 20 % | заказ 1006 | в допустимом диапазоне, но требует бизнес-проверки |

**Итог.** Из 6 заказов валидны 3 (**1001, 1002, 1004**), то есть 50 % выборки. Выборка очень мала, поэтому статистика по ней ненадёжна. Перед аналитикой нужно уточнить у источника данные по заказам 1003, 1005 и 1006 и настроить проверку ключей при объединении файлов.

## Дополнительное задание. Универсальная проверка массива

In [14]:
def check_array(a):
    a = np.asarray(a)
    info = {"shape": a.shape, "dtype": str(a.dtype),
            "numeric": bool(np.issubdtype(a.dtype, np.number))}
    if not info["numeric"]:
        info["note"] = "не числовой массив: проверки NaN/inf/отрицательных пропущены"
        return info

    is_float = np.issubdtype(a.dtype, np.floating)
    nan_mask = np.isnan(a) if is_float else np.zeros(a.shape, bool)
    inf_mask = np.isinf(a) if is_float else np.zeros(a.shape, bool)
    neg_mask = a < 0          # NaN < 0 даёт False, поэтому NaN не считается отрицательным

    info.update(
        nan_count=int(nan_mask.sum()),
        inf_count=int(inf_mask.sum()),
        negative_count=int(neg_mask.sum()),
        nan_idx=np.argwhere(nan_mask).tolist(),
        negative_idx=np.argwhere(neg_mask).tolist(),
    )
    return info

print(check_array(unit_price))
print(check_array(quantity))
print(check_array(broken))
print(check_array(np.array([1.0, np.inf, -np.inf])))
print(check_array(np.array(["a", "b"])))

{'shape': (6,), 'dtype': 'float64', 'numeric': True, 'nan_count': 1, 'inf_count': 0, 'negative_count': 0, 'nan_idx': [[5]], 'negative_idx': []}
{'shape': (6,), 'dtype': 'int64', 'numeric': True, 'nan_count': 0, 'inf_count': 0, 'negative_count': 1, 'nan_idx': [], 'negative_idx': [[4]]}
{'shape': (4, 6), 'dtype': 'float64', 'numeric': True, 'nan_count': 1, 'inf_count': 0, 'negative_count': 1, 'nan_idx': [[1, 2]], 'negative_idx': [[3, 4]]}
{'shape': (3,), 'dtype': 'float64', 'numeric': True, 'nan_count': 0, 'inf_count': 2, 'negative_count': 1, 'nan_idx': [], 'negative_idx': [[2]]}
{'shape': (2,), 'dtype': '<U1', 'numeric': False, 'note': 'не числовой массив: проверки NaN/inf/отрицательных пропущены'}


**Вывод.** Функция векторизована (маски и `sum`, без циклов), работает для массивов любой размерности, возвращает форму, тип, число и позиции `NaN`, `inf` и отрицательных значений. Для нечисловых массивов проверки пропускаются, так как `np.isnan` на строках вызывает ошибку.